# LingBot-Map: replicating upstream's *actual* demo configuration

Runs upstream's two demo scenes (`example/loop`, `example/courthouse`) under **both** inference
configurations described in the paper (arXiv 2604.14141), on a rented GPU, using this repo's
`recon/*.py` unmodified.

## Why there are two configs

The README's one-liner (`demo.py --image_folder example/courthouse --mask_sky`) is **not** the
pipeline that produced upstream's published demo videos. Those came from
`demo_render/batch_demo.py`, driven by `demo_render/process_videos.sh`, which is a different
program with different settings.

| | **A · Direct** (`demo.py`) | **B · VO** (`batch_demo.py`) |
| --- | --- | --- |
| paper section | §4.5 "Default Inference Configuration" — every benchmark number | §4.4 VO mode — *"for the large-scale demo videos … we use VO mode"* |
| mode | `streaming` | `windowed` |
| pose-reference window | k = 64 | k = 64 |
| keyframes | fixed, m = 1 | **adaptive optical flow**, 25.0 px, forced every 100 |
| window size | — | 64 keyframes |
| their input | `example/` folders | the source video, `TARGET_FRAMES=4000`, `IMAGE_STRIDE=1` |

The keyframe mechanism (paper §4.4) predicts pose and depth for each incoming frame, measures
optical flow against the most recent keyframe, and promotes the frame only once that flow clears
a threshold. **`demo.py` exposes it nowhere**, and `gct_stream.py` (Direct) does not implement it
at all — it lives only in `gct_stream_window.py`. So the README command is a strictly weaker
configuration than the one behind their clips, and every windowed run this project has logged so
far used fixed intervals instead.

`recon/reconstruct.py` now takes `--flow_threshold` / `--max_non_keyframe_gap` and passes them
through, so config B is reachable for the first time.

## The metric that decides it

Flow mode returns an `is_keyframe` mask, which the run record turns into **`keyframe_frac`**.

- `keyframe_frac` well below 1.0 → frames are dense enough that the selector is skipping some.
  The mechanism is doing its job.
- `keyframe_frac` **= 1.0** → every frame cleared a 25 px flow threshold, so consecutive *inputs*
  are already further apart than upstream's *keyframe* spacing, and there is no densely-tracked
  frame anywhere in between. That is a property of the footage that no config can undo.

Measured on the shipped frames (phase correlation, scaled to the real 518 px width): courthouse
consecutive frames sit **~47 px** apart, loop **~2 px**. So the expectation going in is
`keyframe_frac ≈ 1.0` for courthouse and clearly below it for loop. Stated up front so the run
can contradict it.

## What else it does

A `kv_cache_sliding_window` ladder (16 → 128) under config A, heavy Open3D cleanup of every run,
renders of each cleaned cloud, and pasteable `notes/experiments.md` rows.

> **Runtime → Change runtime type → A100 or L4 first.** On a T4 (Turing) `reconstruct.py` drops
> to fp16 instead of bf16, changing the numeric path as well as the VRAM.

---

## Part 2 — GrandTour EIG-1, the arm with a ground truth

Everything above scores reconstructions with `traj_length_over_extent`, a *self-consistency*
proxy: it catches a collapsed trajectory but cannot tell a good map from a smoothly-wrong one,
and it once scored a visibly terrible run at 2.87.

Part 2 runs the same `recon/*.py` scripts on **GrandTour EIG-1** (ETH Zurich RSL,
arXiv 2602.18164) — an ANYmal-D descending the Eiger, 429 s / 219.7 m of rocky gravel trail and
stairs — which ships a survey-grade **CPT7** GNSS/INS reference. So every run here is scored in
**metres of ATE**, plus a recovered **metres-per-unit scale** measured against that reference
rather than an assumed 1.5 m eye height.

It answers a question the local 8 GB box cannot even pose. Window count is what the paper says
costs accuracy — VO mode "incurs extra alignment error that compounds with the number of
windows" — and there are two ways to cut it:

| lever | costs | reachable on 8 GB? |
| --- | --- | --- |
| `keyframe_interval` ↑ | **keyframe spacing** — the thing that destroyed courthouse | yes |
| `window_size` ↑ | **VRAM only** — geometry untouched (`VRAM ≈ 3.15 + 0.128·ws` GB at 518×294) | no, `ws=24` is the ceiling |

Sweeping `keyframe_interval` alone confounds the two, so this runs them as **two one-factor
sweeps** plus upstream's own adaptive-flow config as the reference point.


## 1 · Which GPU did we get?

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU (A100 or L4)."
P = torch.cuda.get_device_properties(0)
VRAM_GB = P.total_memory / 1e9
CAP = torch.cuda.get_device_capability()
print(f"\n{P.name}  {VRAM_GB:.1f} GB  sm_{CAP[0]}{CAP[1]}  torch {torch.__version__}")
print(f"local box for comparison: RTX 4060 Ti, 8.6 GB, sm_89  ->  {VRAM_GB/8.6:.1f}x the VRAM")

# reconstruct.py picks bf16 on sm_80+ and fp16 below it. The paper specifies bfloat16, so a
# pre-Ampere card is not a replication -- it changes precision at the same time as VRAM.
if CAP[0] < 8:
    print("\nWARNING: pre-Ampere -> fp16, but the paper specifies bfloat16. Not a clean replication.")
if VRAM_GB < 20:
    print("WARNING: <20 GB. kvsw 128 will probably OOM; everything else should fit.")

## 2 · Configuration

Both configs below are transcribed from the paper and from `demo_render/process_videos.sh`. The
only departures are on the **export** side — how points are selected out of the finished
predictions — and they cannot affect geometry or poses:

- `--pixel_stride 2` is a spatial subsample of the exported cloud (upstream's renderer voxelises
  at 1 mm and our cleanup voxelises at 2 cm, so this changes nothing downstream).
- confidence: ours is a percentile, upstream's is absolute (`1.5` in `demo.py`'s viewer, `2.0` in
  their renderer). Set `CONF_ABS = 2.0` to match theirs exactly; left on the percentile by default
  so these runs stay directly comparable to the ones already in `notes/experiments.md`.

`keyframe_interval` is pinned to 1 rather than left on auto. Upstream's auto is
`(n + 319) // 320`; ours is `ceil(n / 240)`. They agree on loop's 237 frames and **disagree** on
courthouse's 286, so auto would have silently changed the thing being measured.

In [ ]:
# Part 1 (loop + courthouse) is the cache-hypothesis work, answered and closed on Aug 6.
# Off by default: it costs ~10 A100 runs before Part 2 ever starts. Set True to re-run it.
RUN_PART1 = False
SCENES     = ["loop", "courthouse"]   # upstream also ships "university" (324 frames)
CHECKPOINT = "lingbot-map.pt"         # paper/benchmark/demo checkpoint

# ── A · Direct: paper sec 4.5, "Default Inference Configuration" ─────────────
# "Direct Output Mode with a local pose-reference window size k=64 and keyframe
#  interval m=1, at a resolution of 518x518 with bfloat16 precision."
PAPER_DIRECT = dict(
    mode="streaming",
    kv_cache_sliding_window=64,      # k
    keyframe_interval=1,             # m
    num_scale_frames=8,
    camera_num_iterations=4,
    image_size=518,
    preprocess_mode="crop",          # demo.load_images hardcodes crop
)

# ── B · VO: paper sec 4.4, parameterised by demo_render/process_videos.sh ────
#   MODE="windowed"  WINDOW_SIZE=64  FLOW_THRESHOLD=25.0
#   MAX_NON_KEYFRAME_GAP=100  IMAGE_STRIDE=1
# overlap_keyframes=-1 means "unset", which is what batch_demo.py passes; the model
# then resolves it to num_scale_frames internally.
PAPER_VO = dict(
    mode="windowed",
    window_size=64,
    overlap_keyframes=-1,
    kv_cache_sliding_window=64,
    num_scale_frames=8,
    keyframe_interval=1,             # ignored once flow_threshold > 0
    flow_threshold=25.0,
    max_non_keyframe_gap=100,
    camera_num_iterations=4,
    image_size=518,
    preprocess_mode="crop",
)

MASK_SKY = {"courthouse": True, "university": True, "loop": False}   # per upstream's README

# ── export (post-inference; cannot affect poses or drift) ───────────────────
CONF_PERCENTILE = 55
CONF_ABS        = None   # set 2.0 for upstream's renderer vis_threshold
PIXEL_STRIDE    = 2
VRAM_FRACTION   = 0.92

# ── cache ladder, config A ─────────────────────────────────────────────────
RUN_SWEEP   = True
SWEEP_SCENE = "courthouse"
SWEEP       = [(16, 1), (16, 2), (24, 1), (32, 1), (64, 1), (128, 1)]

# ── cleanup / output ───────────────────────────────────────────────────────
CLEAN_HEAVY     = True       # std-ratio 2.0->1.5, min-neighbors 12->16
CAMERA_HEIGHT_M = 1.5
SAVE_TO_DRIVE   = False
KEEP_RAW_PLY    = False

VIDEOS    = {}               # {"my_trail": "/content/drive/MyDrive/trail.MOV"}
VIDEO_FPS = 5

if not RUN_PART1:
    SCENES, RUN_SWEEP, VIDEOS = [], False, {}   # every Part 1 loop iterates SCENES, so this
                                                # no-ops them without touching their cells

WORK = "/content/gd"

# ── Part 2 · GrandTour EIG-1 ────────────────────────────────────────────────
RUN_EIG1 = True
EIG_MISSION = "eig-1"                  # or "snow-2" (the low-texture stress case)
EIG_CAMERA  = "zed2i_left_images"      # 14.91 Hz, radtan, 16:9 -> 518x294.
                                       # hdr_front is 10 Hz + equidistant 120 deg + 3:2 (518x350)

# THE WHOLE MISSION: all 6417 frames, 430 s, 219.7 m. Needs 19.6 GB of system RAM for the
# fp32 prediction stack (A100 runtime has ~83 GB; a T4 runtime at 12 GB cannot) and ~25 min
# per run. Set e.g. (40.0, 160.0) to sweep only the fast open descent instead.
EIG_START, EIG_END = 0.0, None

# Sweep A -- window_size at FIXED keyframe density. Isolates the Sim(3) fusion cost:
# keyframe spacing is constant, only window count moves. This is the one nothing has run.
EIG_SWEEP_WS  = [64, 128, 256]
# Sweep B -- keyframe_interval at FIXED window_size. Isolates keyframe spacing.
EIG_SWEEP_KFI = [1, 2, 4, 6, 8, 10]
EIG_WS_FOR_KFI = 128
# C -- upstream's actual demo config; its keyframe_frac is the model's own answer to "what
# interval does this footage deserve", computed on predicted geometry with upstream's metric.
EIG_RUN_FLOW = True

EIG_OVERLAP_KF = 8       # raise to 16 for snow-2
EIG_CONF_ABS   = 1.3     # demo_render/config/outdoor_drive.yaml's vis_threshold
EIG_RPE_DELTA  = 10.0    # metres of GT path per RPE window

print("config loaded")


## 3 · Install

`lingbot-map`'s `pyproject.toml` does not pin torch, so this installs on top of whatever torch
Colab ships and leaves the CUDA stack alone. If pip asks you to restart, do it and re-run from
cell 1 — the clone and downloads are already on disk and get skipped.

FlashInfer is deliberately not installed: it is upstream's attention *kernel*, not a different
attention, and `reconstruct.py` passes `use_sdpa=True` so PyTorch's SDPA computes the same thing.
Every run prints `flashinfer not available`; that line is expected.

In [ ]:
import os, pathlib, subprocess, sys

LINGBOT_SRC = "/content/lingbot-map"
os.environ["LINGBOT_SRC"] = LINGBOT_SRC   # reconstruct.py reads this to import upstream demo.py

if not pathlib.Path(LINGBOT_SRC, ".git").exists():
    !git clone --depth 1 https://github.com/Robbyant/lingbot-map.git {LINGBOT_SRC}
else:
    print("lingbot-map already cloned")

# `!pip` not `%pip`: the line magic does not expand {LINGBOT_SRC}.
!pip install -q -e "{LINGBOT_SRC}[vis]"
# numcodecs: GrandTour ships raw zarr chunks (blosc/lz4) and Colab has no zarr stack.
!pip install -q open3d imageio-ffmpeg numcodecs

import open3d as o3d
print("open3d", o3d.__version__)
print("frames on disk:", {d.name: len(list(d.glob('*.png')))
                          for d in sorted(pathlib.Path(LINGBOT_SRC, "example").iterdir()) if d.is_dir()})

## 4 · The GeologicDome `recon/` scripts (baked in)

**Nothing to upload.** `recon/*.py` is embedded in this notebook as a base64 zip and unpacked to
`/content/recon` by the cell below, so the scripts that run are by construction the ones
committed alongside it. The repo is private, which used to mean a hand-made `recon.zip` from
Downloads — and a stale one fails as `python3 <missing>.py` -> exit 2, which looks exactly like
an argparse error and reaches the cell with its stderr thrown away.

Maintaining it, after any change under `recon/`:

```bash
python colab/embed_recon.py           # re-bake
python colab/embed_recon.py --check   # exit 1 if the notebook is stale -- run before committing
```

Set the `RECON_SRC` env var to override with a live clone or Drive copy.


In [ ]:
# RECON_BLOB -- generated by colab/embed_recon.py, do not edit by hand.
# recon/*.py travels inside this notebook: 10 files, sha256 cfdcd97f218ee278
# Re-bake with `python colab/embed_recon.py` after changing anything in recon/.
import base64, hashlib, io, os, pathlib, zipfile

RECON_SHA = "cfdcd97f218ee2782b9d7007b8a7b081998d608e2ebd96d212a7da2917add88c"
RECON_B64 = """\
..."""

RECON = pathlib.Path(os.environ.get("RECON_SRC", "/content/recon"))
if "RECON_SRC" in os.environ:
    # Escape hatch: point at a live clone or a Drive copy to override the baked-in copy.
    print(f"RECON_SRC set -- using {RECON} instead of the embedded scripts")
else:
    raw = base64.b64decode(RECON_B64)
    got = hashlib.sha256(raw).hexdigest()
    assert got == RECON_SHA, f"embedded blob corrupted: {got} != {RECON_SHA}"
    RECON.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        z.extractall(RECON)
    print(f"unpacked {len(list(RECON.glob('*.py')))} scripts -> {RECON}  "
          f"(sha {RECON_SHA[:16]})")

missing = [n for n in ['calibrate_scale.py', 'clean_cloud.py', 'cloud_to_hfield.py', 'eval_ate.py', 'extract_frames.py', 'fetch_grandtour.py', 'inspect_cloud.py', 'measure_flow.py', 'reconstruct.py', 'view_viser.py'] if not (RECON / n).exists()]
assert not missing, f"missing from RECON: {missing}"
print("  " + ", ".join(sorted(p.name for p in RECON.glob("*.py"))))


## 5 · Checkpoint + sky segmentation

4.63 GB from HuggingFace. `skyseg.onnx` is needed for courthouse (upstream runs that scene with
`--mask_sky`, and their demo pipeline masks sky on every video); it runs on CPU by design.

In [ ]:
import pathlib
from huggingface_hub import hf_hub_download

CKPT_DIR = pathlib.Path("/content/ckpt"); CKPT_DIR.mkdir(exist_ok=True)
CKPT = pathlib.Path(hf_hub_download("robbyant/lingbot-map", CHECKPOINT, local_dir=str(CKPT_DIR)))
print(f"{CKPT}  {CKPT.stat().st_size/1e9:.2f} GB")

SKYSEG = CKPT_DIR / "skyseg.onnx"
# EIG-1 is deliberately not in SCENES, so it must be named here too -- otherwise a
# Part-2-only run silently skips sky masking (reconstruct() only passes --mask_sky when
# the file exists), and an alpine sky becomes points at effectively infinite depth.
NEED_SKY = any(MASK_SKY.get(s) for s in SCENES) or RUN_EIG1
if NEED_SKY and not SKYSEG.exists():
    !curl -sL -o {SKYSEG} https://huggingface.co/JianyuanWang/skyseg/resolve/main/skyseg.onnx
    print(f"skyseg.onnx  {SKYSEG.stat().st_size/1e6:.0f} MB")

# Upstream's demo_render configs name skyseg_batch.onnx (outdoor_drive.yaml, sky_batch_size 64)
# rather than the skyseg.onnx we use. Checked 2026-08-06 rather than assumed: the two are the
# SAME NETWORK -- identical output node names, identical 320x320 geometry, and bit-identical
# outputs on all 7 heads at batch=1. They differ by 40 bytes of graph metadata making the batch
# axis dynamic, which only enables batching our per-frame call path does not use. So there is
# nothing to gain here and a 176 MB download to lose; skyseg.onnx IS upstream's sky model.


## 6 · The runner

Shells out to `recon/reconstruct.py` so the code path is identical to a local run. Output is teed
to a log — a truncated notebook cell is not evidence, and piping this into anything that swallows
the exit code is how an OOM-killed run gets mistaken for a success.

In [ ]:
import json, pathlib, subprocess, sys, time

WORKP = pathlib.Path(WORK)
(WORKP / "runs").mkdir(parents=True, exist_ok=True)
(WORKP / "logs").mkdir(parents=True, exist_ok=True)
RUNS = {}     # tag -> run.json


def frames_dir(scene):
    d = pathlib.Path(LINGBOT_SRC, "example", scene)
    if d.is_dir():
        return d
    d = WORKP / "frames" / scene
    assert d.is_dir(), f"no frames for {scene!r}"
    return d


def reconstruct(scene, tag, base, quiet=True, **over):
    """Run one reconstruction. `base` is PAPER_DIRECT or PAPER_VO; `over` overrides it."""
    cfg = dict(base); cfg.update(over)
    out = WORKP / "runs" / tag
    log = WORKP / "logs" / f"{tag}.log"

    if (out / "run.json").exists():
        rec = json.loads((out / "run.json").read_text())
        print(f"[{tag}] already done, reusing")
        RUNS[tag] = rec
        return rec

    cmd = [sys.executable, str(RECON / "reconstruct.py"),
           "--frames", str(frames_dir(scene)), "--out", str(out),
           "--model_path", str(CKPT),
           "--pixel_stride", str(PIXEL_STRIDE),
           "--vram_fraction", str(VRAM_FRACTION)]
    cmd += (["--conf_threshold", str(CONF_ABS)] if CONF_ABS is not None
            else ["--conf_percentile", str(CONF_PERCENTILE)])
    for k, v in cfg.items():
        if v is not None:
            cmd += [f"--{k}", str(v)]
    if MASK_SKY.get(scene) and SKYSEG.exists():
        cmd += ["--mask_sky", "--skyseg_model", str(SKYSEG)]

    flow = cfg.get("flow_threshold", 0)
    print(f"[{tag}] {cfg['mode']}  kvsw={cfg['kv_cache_sliding_window']}  "
          + (f"flow={flow}px/gap={cfg['max_non_keyframe_gap']}" if flow
             else f"kfi={cfg['keyframe_interval']}")
          + ("  +sky" if MASK_SKY.get(scene) else ""))
    t0 = time.time()
    with open(log, "w") as fh:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             bufsize=1, env={**os.environ, "LINGBOT_SRC": LINGBOT_SRC})
        for line in p.stdout:
            fh.write(line)
            if not quiet:
                sys.stdout.write(line)
        rc = p.wait()

    if rc != 0:
        print(f"[{tag}] FAILED rc={rc} -- tail of {log}:")
        print("".join(log.read_text().splitlines(keepends=True)[-15:]))
        return None

    rec = json.loads((out / "run.json").read_text())
    RUNS[tag] = rec
    kf = rec.get("keyframe_frac")
    print(f"[{tag}] {time.time()-t0:.0f}s  {rec['n_points']:,} pts  "
          f"{rec['peak_vram_gb']:.2f} GB  ratio {rec['traj_length_over_extent']}"
          + (f"  keyframes {rec['n_keyframes']}/{rec['n_frames']} ({kf:.0%})" if kf else ""))
    return rec


for name, path in VIDEOS.items():
    d = WORKP / "frames" / name
    if not d.is_dir():
        subprocess.run([sys.executable, str(RECON / "extract_frames.py"), path, str(d),
                        "--fps", str(VIDEO_FPS)], check=True)
    if name not in SCENES:
        SCENES.append(name)

print("runner ready")

## 7 · Config A — Direct mode (the paper's benchmark configuration)

`traj_length_over_extent` is camera-path length over scene size. It is a drift **detector**, not a
quality score — a 2026-08-05 run scored 2.87 while looking terrible — but a value in the twenties
means the poses have collapsed.

Reference: locally, loop = **3.36** at kvsw 24, courthouse = **24.88** at kvsw 16.

In [ ]:
LOCAL = {"loop": dict(ratio=3.36, kvsw=24), "courthouse": dict(ratio=24.88, kvsw=16)}

for scene in SCENES:
    reconstruct(scene, f"{scene}_direct", PAPER_DIRECT)

print()
for scene in SCENES:
    r = RUNS.get(f"{scene}_direct")
    if r is None:
        print(f"{scene:12s} FAILED"); continue
    base = LOCAL.get(scene, {}).get("ratio")
    note = f"   local {base} at kvsw {LOCAL[scene]['kvsw']}" if base else ""
    print(f"{scene:12s} ratio {r['traj_length_over_extent']:6.2f}   {r['n_points']:>10,} pts   "
          f"{r['peak_vram_gb']:5.2f} GB   {r['fps']:.2f} fps{note}")

## 8 · Config B — VO mode with adaptive flow keyframes

This is upstream's demo-video configuration, reachable for the first time. Read
**`keyframe_frac`** before anything else: it is the direct test of whether these frames are dense
enough for the mechanism to have anything to select from.

`n_windows_stitched` and `window_scale_span` matter too — VO fuses windows by Sim(3) alignment
over their overlap, and the paper is explicit that this *adds* drift at each boundary
(§4.4: *"VO mode incurs extra alignment error that compounds with the number of windows"*). A
scale span far from 1.0 means that alignment failed.

In [ ]:
for scene in SCENES:
    reconstruct(scene, f"{scene}_vo", PAPER_VO)

print()
print(f"{'scene':12} {'ratio':>7} {'keyframes':>16} {'windows':>8} {'scale span':>11} {'peak GB':>8}")
for scene in SCENES:
    r = RUNS.get(f"{scene}_vo")
    if r is None:
        print(f"{scene:12} FAILED"); continue
    kf = r.get("keyframe_frac")
    kf_txt = f"{r['n_keyframes']}/{r['n_frames']} ({kf:.0%})" if kf is not None else "n/a"
    print(f"{scene:12} {r['traj_length_over_extent']:>7.2f} {kf_txt:>16} "
          f"{r['n_windows_stitched']:>8} {r['window_scale_span']:>11.2f} {r['peak_vram_gb']:>8.2f}")

print("\nA vs B, same frames, same card:")
for scene in SCENES:
    a, b = RUNS.get(f"{scene}_direct"), RUNS.get(f"{scene}_vo")
    if a and b:
        print(f"  {scene:12} Direct {a['traj_length_over_extent']:6.2f}  ->  "
              f"VO {b['traj_length_over_extent']:6.2f}")

## 9 · Cache ladder (config A)

Locally this is unanswerable — 24 is the ceiling, so the ladder has one rung.

The pair to read is **`kvsw 16 / kfi 2`** against **`kvsw 32 / kfi 1`**: same span of footage,
half the cached views. If only the view count matters, `(32, 1)` wins and VRAM is the whole lever.
If they land together, the model wants horizon, and buying VRAM will not fix long walks.

In [ ]:
sweep_rows = []
if RUN_SWEEP:
    for kvsw, kfi in SWEEP:
        if kvsw > 64 and VRAM_GB < 20:
            print(f"skipping kvsw {kvsw} on a {VRAM_GB:.0f} GB card"); continue
        if (kvsw, kfi) == (PAPER_DIRECT["kv_cache_sliding_window"],
                           PAPER_DIRECT["keyframe_interval"]):
            r = RUNS.get(f"{SWEEP_SCENE}_direct")        # already paid for in cell 7
        else:
            r = reconstruct(SWEEP_SCENE, f"{SWEEP_SCENE}_kv{kvsw}_kfi{kfi}", PAPER_DIRECT,
                            kv_cache_sliding_window=kvsw, keyframe_interval=kfi)
        sweep_rows.append((kvsw, kfi, r))

    print(f"\n{SWEEP_SCENE}: cache sweep")
    print(f"{'kvsw':>5} {'kfi':>4} {'ratio':>8} {'peak GB':>8} {'fps':>6} {'points':>11}")
    for kvsw, kfi, r in sweep_rows:
        if r is None:
            print(f"{kvsw:>5} {kfi:>4} {'OOM/FAIL':>8}"); continue
        print(f"{kvsw:>5} {kfi:>4} {r['traj_length_over_extent']:>8.2f} {r['peak_vram_gb']:>8.2f} "
              f"{r['fps']:>6.2f} {r['n_points']:>11,}")

## 10 · Verdict

Decided from numbers, not from how the renders feel. `traj_length_over_extent` ≤ 6 is the bar
`calibrate_scale.py` requires before it will trust poses enough to anchor metric scale.

In [ ]:
print("=" * 74)
for scene in SCENES:
    a, b = RUNS.get(f"{scene}_direct"), RUNS.get(f"{scene}_vo")
    if not (a and b):
        continue
    best = min(a["traj_length_over_extent"], b["traj_length_over_extent"])
    kf = b.get("keyframe_frac")
    print(f"\n{scene}:  Direct {a['traj_length_over_extent']:.2f}   "
          f"VO+flow {b['traj_length_over_extent']:.2f}   best {best:.2f}")
    if kf is not None:
        if kf >= 0.99:
            print(f"  keyframe_frac {kf:.0%} -- EVERY frame cleared the 25 px flow threshold.")
            print("  These frames are sampled more sparsely than upstream's own keyframe")
            print("  spacing, so there is no densely-tracked frame in between and the")
            print("  selector has nothing to select. This is an input property, not a config.")
        else:
            print(f"  keyframe_frac {kf:.0%} -- the selector is genuinely skipping frames,")
            print("  so the footage is inside the regime the mechanism was built for.")
    print("  => " + ("USABLE (ratio <= 6, scale can be anchored)" if best <= 6
                     else "COLLAPSED (ratio > 6; calibrate_scale.py will refuse this run)"))

r16 = next((r["traj_length_over_extent"] for k, f, r in sweep_rows if (k, f) == (16, 1) and r), None)
ch = RUNS.get("courthouse_direct")
if ch and r16:
    print(f"\ncache ladder control: kvsw 16 here = {r16:.2f}, on the local 8 GB box = 24.88")
    print(f"                     kvsw 64 here = {ch['traj_length_over_extent']:.2f}")
    print("  If those two agree, the card was never the variable.")
print("=" * 74)

## 11 · Heavy Open3D cleanup

Two scripts, unmodified:

- **`calibrate_scale.py`** — fits candidate ground planes, keeps the one holding camera height
  *constant* (inlier count picks a wall in a corridor), divides median camera height by an assumed
  1.5 m eye height. It **refuses** above drift ratio 6, because drifted poses cannot anchor
  anything. Whether courthouse now clears that gate is itself a result.
- **`clean_cloud.py --scale auto`** — scales to metres first so every filter size is a real
  distance, then statistical outlier removal → radius filtering (the flying-pixel streaks shed at
  occlusion edges, which statistical removal misses because each streak is locally dense along its
  own filament) → 2 cm voxel downsample → ground plane to +Z at z=0.

In [ ]:
import json, subprocess, sys

MAIN_TAGS = [f"{s}_{k}" for s in SCENES for k in ("direct", "vo")]


def clean(tag):
    d = WORKP / "runs" / tag
    if not (d / "cloud.ply").exists():
        return None
    if (d / "clean_stats.json").exists():
        print(f"[{tag}] already cleaned"); return json.loads((d / "clean_stats.json").read_text())

    cal = subprocess.run([sys.executable, str(RECON / "calibrate_scale.py"), str(d),
                          "--camera-height", str(CAMERA_HEIGHT_M)], capture_output=True, text=True)
    print(f"── {tag}: scale")
    print(cal.stdout.strip())
    if cal.returncode != 0:
        tail = cal.stderr.strip().splitlines()
        print("  REFUSED:", tail[-1] if tail else f"rc={cal.returncode}")

    cmd = [sys.executable, str(RECON / "clean_cloud.py"), str(d)]
    if (d / "scale.json").exists():
        cmd += ["--scale", "auto"]
    else:
        print("  no scale.json -> cleaning in arbitrary units (NOT terrain-ready)")
    if CLEAN_HEAVY:
        cmd += ["--std-ratio", "1.5", "--min-neighbors", "16"]

    cl = subprocess.run(cmd, capture_output=True, text=True)
    print(f"── {tag}: clean"); print(cl.stdout.strip() or cl.stderr.strip())
    return json.loads((d / "clean_stats.json").read_text()) if cl.returncode == 0 else None


cleaned = {t: clean(t) for t in MAIN_TAGS if t in RUNS}

## 12 · Look at all four maps

Direct and VO side by side for each scene. Tries `inspect_cloud.py` first (four orbit views
through Open3D's headless EGL path, plus `inspect_stats.json`); Colab does not always expose EGL
to Open3D, so there is a matplotlib fallback that plots the same four views.

In [ ]:
import numpy as np, subprocess, sys
import matplotlib.pyplot as plt
from IPython.display import Image, display


def mpl_preview(ply, title, max_pts=250_000):
    pcd = o3d.io.read_point_cloud(str(ply))
    p, c = np.asarray(pcd.points), np.asarray(pcd.colors)
    if len(p) > max_pts:
        i = np.random.default_rng(0).choice(len(p), max_pts, replace=False)
        p, c = p[i], (c[i] if len(c) else c)
    c = c if len(c) else np.full((len(p), 3), 0.35)

    th = np.deg2rad(45)
    rot = p @ np.array([[np.cos(th), -np.sin(th), 0], [np.sin(th), np.cos(th), 0], [0, 0, 1]]).T
    views = [("top  (x,y)", p[:, 0], p[:, 1]), ("front (x,z)", p[:, 0], p[:, 2]),
             ("side  (y,z)", p[:, 1], p[:, 2]), ("oblique", rot[:, 0], rot[:, 2])]

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    for ax, (name, u, v) in zip(axes.ravel(), views):
        ax.scatter(u, v, c=np.clip(c, 0, 1), s=0.06, linewidths=0)
        ax.set_aspect("equal"); ax.set_title(name, fontsize=10); ax.tick_params(labelsize=7)
    fig.suptitle(title, fontsize=13); fig.tight_layout(); plt.show()


for tag in MAIN_TAGS:
    d = WORKP / "runs" / tag
    ply = d / "cloud_clean.ply"
    if not ply.exists():
        continue
    st = json.loads((d / "clean_stats.json").read_text())
    r = RUNS[tag]
    print(f"\n{'='*74}\n{tag}:  ratio {r['traj_length_over_extent']}   "
          f"{st['n_out']:,} points   extent {st['extent_out']} {st['units']}"
          + (f"   keyframes {r['keyframe_frac']:.0%}" if r.get("keyframe_frac") else "")
          + f"\n{'='*74}")

    shown = False
    try:
        res = subprocess.run([sys.executable, str(RECON / "inspect_cloud.py"), str(d),
                              "--clean", "--tag", "clean_"], capture_output=True, text=True,
                             timeout=1200)
        pngs = sorted(d.glob("view_clean_*.png"))
        if res.returncode == 0 and pngs:
            for q in pngs:
                display(Image(filename=str(q), width=760))
            shown = True
    except Exception as e:
        print("inspect_cloud unavailable:", type(e).__name__)
    if not shown:
        print("(Open3D offscreen EGL unavailable -- matplotlib fallback)")
        mpl_preview(ply, tag)

## 13 · EIG-1 — fetch the mission and its CPT7 ground truth

`recon/fetch_grandtour.py` pulls the mission from HuggingFace (zarr + JPEG tars, no
registration), rectifies the released camera model to an explicit pinhole **straight into the
518-wide output raster** — one resampling, not two — and composes the ground truth through the
camera's 0.417 m lever arm off the CPT7. It writes `frames/`, `gt_tum.txt`, `mission.json` and a
contact sheet.

Two release details it encodes, both easy to get wrong: the `hdr_front` stream is **10 Hz, not
the paper's 30 fps** (its own zarr attrs say so), and the zarr chunks are zero-padded to their
declared chunk shape, so decoding the whole ground-truth tar would inflate `pose_cov` to 2.4 GB
of zeros.

In [ ]:
import json, pathlib, subprocess, sys
from IPython.display import Image, display

EIG_DIR = WORKP / "grandtour" / EIG_MISSION
FRAME_LINK = WORKP / "frames" / EIG_MISSION       # so frames_dir() finds it


def run_script(script, *args):
    """Run a recon/ script, streaming its output into the cell.

    subprocess stderr does not reach a Colab cell, so `check=True` raises a bare
    CalledProcessError with nothing readable in it -- and python exits 2 when it cannot
    even open the script, which looks identical to an argparse error. Both failure modes
    are named explicitly here instead.
    """
    exe = RECON / script
    if not exe.exists():
        raise SystemExit(
            f"{exe} is missing.\n\n"
            f"recon/*.py is baked into this notebook, so this should not happen unless\n"
            f"RECON_SRC points somewhere incomplete, or cell 4 was not run.\n\n"
            f"  RECON = {RECON}\n"
            f"  currently present: {sorted(q.name for q in RECON.glob('*.py'))}\n\n"
            f"Run cell 4 (the RECON_BLOB cell). If recon/ changed on disk, re-bake with\n"
            f"`python colab/embed_recon.py` and pull the notebook again.")

    cmd = [sys.executable, "-u", str(exe), *[str(a) for a in args]]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        sys.stdout.write(line)
        tail.append(line)
        del tail[:-40]
    if proc.wait() != 0:
        raise SystemExit(f"{script} failed (exit {proc.returncode}). Last lines:\n"
                         + "".join(tail))


if RUN_EIG1 and not (EIG_DIR / "mission.json").exists():
    args = ["--mission", EIG_MISSION, "--camera", EIG_CAMERA,
            "--cache", WORKP / "grandtour_cache", "--out", EIG_DIR,
            "--start", EIG_START]
    if EIG_END is not None:
        args += ["--end", EIG_END]
    run_script("fetch_grandtour.py", *args)

if RUN_EIG1:
    FRAME_LINK.parent.mkdir(parents=True, exist_ok=True)
    if not FRAME_LINK.exists():
        FRAME_LINK.symlink_to(EIG_DIR / "frames", target_is_directory=True)
    MASK_SKY[EIG_MISSION] = True          # alpine scene: sky is at effectively infinite depth

    MI = json.loads((EIG_DIR / "mission.json").read_text())
    print(f"\n{MI['mission_short']} -> {MI['mission_folder']}  ({MI['camera']})")
    print(f"  source  {MI['source']['width']}x{MI['source']['height']} "
          f"{MI['source']['distortion_model']} @ {MI['source']['rate_hz']} Hz")
    print(f"  output  {MI['output']['width']}x{MI['output']['height']} pinhole, "
          f"hFOV {MI['output']['hfov_deg']:.1f} deg, {MI['output']['n_frames']} frames")
    print(f"  GT      {MI['ground_truth']['n_poses']} poses, "
          f"{MI['ground_truth']['path_length_m']:.1f} m of path, "
          f"lever arm {MI['ground_truth']['lever_arm_m']} m")
    # Look at the frames once: this catches a wrong camera model or the upside-down ZED2i
    # mount silently getting through, which no downstream metric would flag.
    display(Image(filename=str(EIG_DIR / "contact_sheet.jpg"), width=900))

## 14 · Preflight — is this footage dense enough to reconstruct at all?

Upstream's keyframe selector promotes a frame once **mean dense flow magnitude** clears
**25.0 px** at 518 px width (`process_videos.sh`), with every intermediate frame densely
tracked. `example/courthouse` failed because its *consecutive* frames were already ~47 px apart —
past upstream's *keyframe* spacing, with nothing in between. That is an input property no config
can undo, so it is worth measuring before spending a GPU-hour.

`measure_flow.py` computes the same physical quantity with Farneback on the CPU. Read the
interval whose median lands near 25 px — but treat the top of the curve with suspicion, because
past a few hundred milliseconds the estimator decorrelates and quietly under-reports.

In [ ]:
import json

FLOW = None
if RUN_EIG1:
    fj = EIG_DIR / "flow.json"
    if not fj.exists():
        run_script("measure_flow.py",
                   "--frames", EIG_DIR / "frames",
                   "--fps", MI["output"]["effective_hz"],
                   "--sample", 400, "--out", fj)
    FLOW = json.loads(fj.read_text())

### Chart style

One place defines the surface, palette and mark specs for every figure below, so the whole
notebook reads as one system. The figures carry their **own light surface** rather than
inheriting Colab's theme — a screenshot pasted into the lab notebook then looks the same
whichever theme it was captured under.

Categorical hues are assigned in fixed order and never cycled, and every series is direct-labeled
as well as legended, so identity never depends on colour alone.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Categorical slots 1-3, in fixed order. Validated all-pairs (worst CVD dE 9.2, normal-vision
# 24.0) -- which is also why this notebook never puts more than three series on one axis.
C_BLUE, C_ORANGE, C_AQUA = "#2a78d6", "#eb6834", "#1baf7a"
SERIES = [C_BLUE, C_ORANGE, C_AQUA]
SURFACE, INK, INK_2, INK_MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#a8a69c"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": INK_MUTED, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e8e7e2", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.titlepad": 10, "font.size": 10,
    "legend.frameon": False, "lines.linewidth": 2.0, "lines.markersize": 7,
})


def label_end(ax, x, y, text, color, dx=6):
    """Direct label at a line's end -- text wears an ink token, never the series colour."""
    ax.annotate(text, xy=(x, y), xytext=(dx, 0), textcoords="offset points",
                color=INK_2, fontsize=9, va="center", fontweight="medium")


def show(fig):
    fig.tight_layout()
    plt.show()


print("chart style loaded")

In [ ]:
if FLOW:
    rows = FLOW["intervals"]
    k = [r["keyframe_interval"] for r in rows]

    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    # The threshold is a reference line, not a series -- it gets no categorical slot.
    ax.axhline(FLOW["upstream_flow_target_px"], color=INK_MUTED, lw=1.5, ls="--", zorder=1)
    ax.annotate("upstream keyframe target · 25 px", xy=(k[-1], 25), xytext=(0, 6),
                textcoords="offset points", ha="right", color=INK_2, fontsize=9)

    # Direct labels must be DISTINCT -- "dense flow" for two different series is no label
    # at all. Short tags here, full names in the legend.
    for (key, name, tag, c) in [
            ("flow_median_px", "dense flow (median)", "median", C_BLUE),
            ("flow_p90_px", "dense flow (p90)", "p90", C_ORANGE),
            ("phase_shift_median_px", "phase-correlation shift", "phase corr", C_AQUA)]:
        v = [r[key] for r in rows]
        ax.plot(k, v, "-o", color=c, label=name, zorder=3,
                markeredgecolor=SURFACE, markeredgewidth=1.5)
        label_end(ax, k[-1], v[-1], tag, c)

    ax.set_xlabel("--keyframe_interval")
    ax.set_ylabel("displacement at 518 px (px)")
    ax.set_title(f"Frame spacing · {EIG_MISSION} · {FLOW['n_frames']} frames @ {FLOW['fps']:.2f} Hz")
    ax.set_xticks(k)
    ax.set_xlim(min(k) - 0.3, max(k) + 1.9)
    ax.set_ylim(0, max(28, max(r["flow_p90_px"] for r in rows) * 1.15))
    # Legend below the axes: inside, it collides with whichever series ends lowest.
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=3)
    show(fig)

    print(f"\nrecommended --keyframe_interval {FLOW['recommended_keyframe_interval']}")
    print("Caveat: if the dense-flow curve flattens while the phase-correlation curve keeps")
    print("climbing, the estimator is decorrelating and the top of the range is a LOWER BOUND.")
    print("keyframe_frac from the flow-mode run below settles it on real predicted geometry.")

## 15 · The two one-factor sweeps, scored against CPT7

- **A — `window_size` ∈ {64, 128, 256} at `keyframe_interval` 1.** Keyframe spacing is held
  constant, so only window count moves. Isolates the paper's compounding-alignment claim.
- **B — `keyframe_interval` ∈ {1, 2, 4, 6, 8, 10} at `window_size` 128.** Isolates keyframe
  spacing. This is the original plan, now with A to disentangle it from.
- **C — `--flow_threshold 25.0`**, upstream's own demo configuration.

Each run is scored by `recon/eval_ate.py`: Umeyama Sim(3) ATE, RPE over a fixed metric window
(ATE alone is dominated by wherever the trajectory diverges worst, so it cannot separate one late
failure from a uniform wobble), a per-segment breakdown, and the recovered metres-per-unit.

In [ ]:
import json, subprocess, sys, time

GT = EIG_DIR / "gt_tum.txt"
EIG_RUNS = {}      # tag -> {**run.json, **ate.json, sweep, window_size, keyframe_interval}


def score(tag, out):
    """eval_ate.py on a finished run -> merged record."""
    if not (RECON / "eval_ate.py").exists():
        raise SystemExit("recon/eval_ate.py is missing -- run cell 4 (RECON_BLOB); if "
                         "recon/ changed on disk, re-bake with colab/embed_recon.py")
    r = subprocess.run([sys.executable, str(RECON / "eval_ate.py"),
                        "--run", str(out), "--gt", str(GT),
                        "--rpe_delta", str(EIG_RPE_DELTA), "--plot"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"[{tag}] eval_ate FAILED:\n{r.stdout[-800:]}{r.stderr[-800:]}")
        return None
    return json.loads((out / "ate.json").read_text())


def run_eig(tag, sweep, **over):
    cfg = dict(PAPER_VO)
    cfg.update(dict(window_size=EIG_WS_FOR_KFI, overlap_keyframes=EIG_OVERLAP_KF,
                    flow_threshold=0.0, keyframe_interval=1))
    cfg.update(over)
    rec = reconstruct(EIG_MISSION, tag, cfg, **{})
    if rec is None:
        return None
    ate = score(tag, WORKP / "runs" / tag)
    if ate is None:
        return None
    EIG_RUNS[tag] = {**rec, **ate, "sweep": sweep,
                     "window_size": cfg["window_size"],
                     "keyframe_interval": (0 if cfg.get("flow_threshold") else
                                           cfg["keyframe_interval"])}
    print(f"      -> ATE {ate['ate_sim3_rmse_m']:.3f} m ({ate['ate_pct_of_path']:.2f}% of path), "
          f"scale {ate['scale_m_per_unit']:.4f} m/unit, "
          f"{rec['n_windows_stitched']} windows, span {rec['window_scale_span']}x")
    return EIG_RUNS[tag]


if RUN_EIG1:
    # Absolute confidence cut, not our percentile: a percentile keeps a fixed FRACTION of every
    # run, which is exactly wrong when the point is comparing runs to each other.
    CONF_ABS, _CONF_SAVED = EIG_CONF_ABS, CONF_ABS

    t0 = time.time()
    for ws in EIG_SWEEP_WS:                                   # A
        run_eig(f"{EIG_MISSION}_A_ws{ws}", "A · window_size", window_size=ws, keyframe_interval=1)
    for kfi in EIG_SWEEP_KFI:                                 # B
        run_eig(f"{EIG_MISSION}_B_kfi{kfi}", "B · keyframe_interval",
                window_size=EIG_WS_FOR_KFI, keyframe_interval=kfi)
    if EIG_RUN_FLOW:                                          # C
        run_eig(f"{EIG_MISSION}_C_flow", "C · adaptive flow",
                window_size=EIG_WS_FOR_KFI, flow_threshold=25.0, max_non_keyframe_gap=100)

    CONF_ABS = _CONF_SAVED
    print(f"\n{len(EIG_RUNS)} runs in {(time.time()-t0)/60:.1f} min")

## 16 · Results

The table is the accessible view of every chart below it — and the relief for the one palette
slot that sits under 3:1 on this surface.

In [ ]:
import pandas as pd

if EIG_RUNS:
    df = pd.DataFrame([{
        "run": t, "sweep": r["sweep"],
        "ws": r["window_size"],
        "kfi": ("flow" if r["keyframe_interval"] == 0 else r["keyframe_interval"]),
        "windows": r["n_windows_stitched"],
        "scale span": r["window_scale_span"],
        "keyframe %": (round(100 * r["keyframe_frac"]) if r.get("keyframe_frac") else None),
        "ATE (m)": r["ate_sim3_rmse_m"],
        "ATE (% path)": r["ate_pct_of_path"],
        f"RPE@{EIG_RPE_DELTA:g}m (m)": r["rpe_rmse_m"],
        "scale (m/unit)": r["scale_m_per_unit"],
        "ratio": r["traj_length_over_extent"],
        "VRAM (GB)": r["peak_vram_gb"],
    } for t, r in EIG_RUNS.items()]).sort_values(["sweep", "ws", "kfi"])

    display(df.style.format(precision=3).background_gradient(
        subset=["ATE (m)"], cmap="Blues").hide(axis="index"))

    best = df.loc[df["ATE (m)"].idxmin()]
    print(f"\nbest: {best['run']}  ATE {best['ATE (m)']:.3f} m "
          f"({best['ATE (% path)']:.2f}% of path) with {best['windows']} windows")
    print(f"CPT7 reference carries ~0.132 m mean ATE of its own -- anything near that is at the")
    print("noise floor of the ground truth, not better than it.")

In [ ]:
if EIG_RUNS:
    from matplotlib.lines import Line2D

    A = sorted([r for r in EIG_RUNS.values() if r["sweep"].startswith("A")],
               key=lambda r: r["n_windows_stitched"])
    B = sorted([r for r in EIG_RUNS.values() if r["sweep"].startswith("B")],
               key=lambda r: r["n_windows_stitched"])
    C = [r for r in EIG_RUNS.values() if r["sweep"].startswith("C")]
    ARMS = [(A, "A · window_size", "window_size", C_BLUE),
            (B, "B · keyframe_interval", "keyframe_interval", C_ORANGE)]

    def pad_right(ax, frac=0.26):
        """Direct labels sit outside the last marker; without this they clip at the spine."""
        lo, hi = ax.get_xlim()
        ax.set_xlim(lo, hi + frac * (hi - lo))

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.4))

    # LEFT -- the money chart. Both levers on ONE axis (window count), so "does cutting
    # windows help, and does it matter HOW you cut them" is read directly.
    # RIGHT -- the mechanism: per-window scale disagreement, what Sim(3) fusion gets wrong.
    for ax, key, ylab, title, ref, reflab in [
            (axes[0], "ate_sim3_rmse_m", "ATE, Sim(3) aligned (m)",
             "Does cutting window count help — and does it matter how?",
             0.132, "CPT7 reference noise floor · 0.132 m"),
            (axes[1], "window_scale_span", "per-window scale span (max/min)",
             "The mechanism — how far apart the windows' scales land",
             1.0, "windows agree · 1.0x")]:
        for arm, _name, tag, c in ARMS:
            if not arm:
                continue
            x = [r["n_windows_stitched"] for r in arm]
            y = [r[key] for r in arm]
            ax.plot(x, y, "-o", color=c, markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=3)
            label_end(ax, x[-1], y[-1], tag, c)
        for r in C:
            ax.plot(r["n_windows_stitched"], r[key], "*", color=C_AQUA, markersize=17,
                    markeredgecolor=SURFACE, markeredgewidth=1.2, zorder=4)
            label_end(ax, r["n_windows_stitched"], r[key], "adaptive flow", C_AQUA)
        ax.axhline(ref, color=INK_MUTED, lw=1.5, ls="--", zorder=1)
        ax.annotate(reflab, xy=(0.98, ref), xycoords=("axes fraction", "data"),
                    xytext=(0, 5), textcoords="offset points", ha="right",
                    color=INK_2, fontsize=9)
        ax.set_xlabel("windows stitched")
        ax.set_ylabel(ylab)
        ax.set_title(title, fontsize=11.5)
        pad_right(ax)

    # One figure-level legend: a per-axes legend left the right-hand panel with none.
    handles = [Line2D([], [], color=c, marker="o", lw=2, markeredgecolor=SURFACE,
                      markeredgewidth=1.5, label=name) for _a, name, _t, c in ARMS]
    handles.append(Line2D([], [], color=C_AQUA, marker="*", lw=0, markersize=15,
                          markeredgecolor=SURFACE, label="C · adaptive flow"))
    fig.legend(handles=handles, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.06))
    show(fig)

In [ ]:
import numpy as np

def load_pair(tag):
    """(GT xyz, Sim(3)-aligned estimate xyz) for one run, associated by source frame index."""
    sys.path.insert(0, str(RECON))
    from eval_ate import load_gt, load_est, umeyama
    gi_, G_ = load_gt(GT)
    ei_, E_ = load_est(WORKP / "runs" / tag)
    common, gi, ei = np.intersect1d(gi_, ei_, return_indices=True)
    G, E = G_[gi], E_[ei]
    s, R, t = umeyama(E, G, with_scale=True)
    return G, (s * (R @ E.T)).T + t


if EIG_RUNS:
    tags = list(EIG_RUNS)
    ncol = 3
    nrow = int(np.ceil(len(tags) / ncol))

    # Small multiples: one panel per run, two series each (GT vs estimate). Same two hues in
    # every panel, so the reader learns the mapping once.
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 4.3 * nrow))
    for ax, tag in zip(np.ravel(axes), tags):
        G, E = load_pair(tag)
        ax.plot(G[:, 0], G[:, 1], color=C_BLUE, lw=2.4, label="CPT7 ground truth", zorder=2)
        ax.plot(E[:, 0], E[:, 1], color=C_ORANGE, lw=1.4, label="LingBot-Map", zorder=3)
        ax.set_aspect("equal")
        r = EIG_RUNS[tag]
        kfi = "flow" if r["keyframe_interval"] == 0 else f"kfi {r['keyframe_interval']}"
        ax.set_title(f"ws {r['window_size']} · {kfi}\nATE {r['ate_sim3_rmse_m']:.2f} m · "
                     f"{r['n_windows_stitched']} windows", fontsize=10)
        ax.tick_params(labelsize=8)
    for ax in np.ravel(axes)[len(tags):]:
        ax.axis("off")
    h, l = np.ravel(axes)[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.015))
    fig.suptitle("Trajectory vs CPT7, top-down · east/north (m)", fontsize=13,
                 fontweight="bold", x=0.02, ha="left", color=INK)
    show(fig)

In [ ]:
if EIG_RUNS:
    # Error against distance travelled. If Sim(3) fusion is the dominant error term this is a
    # SAWTOOTH whose humps sit on window boundaries -- which is a different diagnosis, and a
    # different fix, from error that grows smoothly.
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.4 * nrow), sharex=True)
    for ax, tag in zip(np.ravel(axes), tags):
        G, E = load_pair(tag)
        d = np.concatenate([[0.0], np.cumsum(np.linalg.norm(np.diff(G, axis=0), axis=1))])
        err = np.linalg.norm(E - G, axis=1)
        r = EIG_RUNS[tag]
        # Expected boundaries: windows are evenly spaced in FRAMES, so evenly in path only
        # roughly -- close enough to see whether the humps line up.
        for b in range(1, r["n_windows_stitched"]):
            ax.axvline(d[-1] * b / r["n_windows_stitched"], color=INK_MUTED, lw=1, ls=":", zorder=1)
        ax.plot(d, err, color=C_BLUE, lw=1.4, zorder=3)     # single series -> no legend box
        ax.axhline(r["ate_sim3_rmse_m"], color=INK_MUTED, lw=1.2, ls="--", zorder=2)
        kfi = "flow" if r["keyframe_interval"] == 0 else f"kfi {r['keyframe_interval']}"
        ax.set_title(f"ws {r['window_size']} · {kfi} · {r['n_windows_stitched']} windows", fontsize=10)
        ax.tick_params(labelsize=8)
    for ax in np.ravel(axes)[len(tags):]:
        ax.axis("off")
    fig.suptitle("Position error vs distance · dotted = expected window boundaries, "
                 "dashed = that run's RMSE", fontsize=12, fontweight="bold", x=0.02, ha="left",
                 color=INK)
    fig.supxlabel("distance along GT path (m)", color=INK_2, fontsize=10)
    fig.supylabel("position error (m)", color=INK_2, fontsize=10)
    show(fig)

## 17 · The map itself

Numbers say whether the *trajectory* is right; they say much less about whether the surface a
policy would train on is. Below: the Open3D offscreen renders for the best run, then an
interactive 3D view you can actually rotate — drag to orbit, scroll to zoom.

In [ ]:
if EIG_RUNS:
    BEST = min(EIG_RUNS, key=lambda t: EIG_RUNS[t]["ate_sim3_rmse_m"])
    bd = WORKP / "runs" / BEST
    print(f"best run: {BEST}  ATE {EIG_RUNS[BEST]['ate_sim3_rmse_m']:.3f} m\n")

    # eval_ate.py already wrote this one during scoring.
    if (bd / "ate_plot.png").exists():
        display(Image(filename=str(bd / "ate_plot.png"), width=980))

    res = subprocess.run([sys.executable, str(RECON / "inspect_cloud.py"), str(bd),
                          "--voxel", "0.01"], capture_output=True, text=True, timeout=1800)
    pngs = sorted(bd.glob("view_*.png"))
    if res.returncode == 0 and pngs:
        for q in pngs:
            display(Image(filename=str(q), width=820))
    else:
        print("(Open3D offscreen unavailable -- the interactive view below still works)")
        print(res.stdout[-600:], res.stderr[-400:])

In [ ]:
if EIG_RUNS:
    import numpy as np, plotly.graph_objects as go

    pcd = o3d.io.read_point_cloud(str(bd / "cloud.ply"))
    P_ = np.asarray(pcd.points)
    C_ = np.asarray(pcd.colors)
    # Plotly is fine up to a few hundred thousand markers in a browser; beyond that the
    # notebook gets sluggish for no extra readable detail.
    MAXP = 120_000
    if len(P_) > MAXP:
        i = np.random.default_rng(0).choice(len(P_), MAXP, replace=False)
        P_, C_ = P_[i], (C_[i] if len(C_) else C_)
    col = (["rgb(%d,%d,%d)" % tuple((c * 255).astype(int)) for c in np.clip(C_, 0, 1)]
           if len(C_) else C_BLUE)

    traj = np.load(bd / "trajectory.npz")["cam_centers"]

    fig = go.Figure([
        go.Scatter3d(x=P_[:, 0], y=P_[:, 1], z=P_[:, 2], mode="markers",
                     marker=dict(size=1.1, color=col, opacity=0.85),
                     name="point cloud", hoverinfo="skip"),
        go.Scatter3d(x=traj[:, 0], y=traj[:, 1], z=traj[:, 2], mode="lines",
                     line=dict(color=C_ORANGE, width=5), name="camera path"),
    ])
    fig.update_layout(
        title=dict(text=f"{BEST} — {len(np.asarray(pcd.points)):,} points "
                        f"(showing {len(P_):,}) · drag to orbit",
                   x=0.02, xanchor="left", font=dict(size=14, color=INK)),
        scene=dict(aspectmode="data",
                   xaxis=dict(title="x", backgroundcolor=SURFACE, gridcolor="#e8e7e2"),
                   yaxis=dict(title="y", backgroundcolor=SURFACE, gridcolor="#e8e7e2"),
                   zaxis=dict(title="z", backgroundcolor=SURFACE, gridcolor="#e8e7e2")),
        paper_bgcolor=SURFACE, height=680, margin=dict(l=0, r=0, t=52, b=0),
        legend=dict(orientation="h", yanchor="bottom", y=0.01, x=0.02))
    fig.show()

In [ ]:
if EIG_RUNS:
    print("=" * 78)
    print("paste into notes/experiments.md:\n")
    gpu_ = P.name.replace("NVIDIA ", "")
    for tag, r in EIG_RUNS.items():
        kfi = ("**flow 25 px**" if r["keyframe_interval"] == 0
               else f"kfi={r['keyframe_interval']}")
        cfg_ = (f"LingBot-Map **{CHECKPOINT.replace('.pt','')}** on **Colab {gpu_} {VRAM_GB:.0f} GB**, "
                f"GrandTour **{EIG_MISSION}** {EIG_CAMERA} ({r['n_frames']} frames, "
                f"{EIG_START:g}-{'end' if EIG_END is None else format(EIG_END, 'g')} s), "
                f"windowed ws={r['window_size']} nsf=8 overlap_kf={EIG_OVERLAP_KF}, {kfi}, "
                f"conf {EIG_CONF_ABS} abs, --mask_sky, 518x294")
        met = (f"**ATE {r['ate_sim3_rmse_m']:.3f} m ({r['ate_pct_of_path']:.2f}% of "
               f"{r['gt_path_length_m']:.1f} m)**, RPE@{EIG_RPE_DELTA:g}m {r['rpe_rmse_m']}, "
               f"scale {r['scale_m_per_unit']:.4f} m/unit, {r['n_windows_stitched']} windows, "
               f"span {r['window_scale_span']}x, ratio {r['traj_length_over_extent']}, "
               f"{r['peak_vram_gb']:.2f} GB"
               + (f", keyframe_frac {r['keyframe_frac']:.0%}" if r.get("keyframe_frac") else ""))
        print(f"| {STAMP if 'STAMP' in dir() else '<date>'}-colab-{tag.replace('_','-')} "
              f"| <hash> | {cfg_} | — | {met} | <takeaway> |")
    print("=" * 78)

## 13 · Take the results home

Zips per-run artifacts and prints `notes/experiments.md` rows. Rows are mandatory per `CLAUDE.md`
— every reconstruction run gets one, failures included. Fill in `commit` with the short hash of
the commit this notebook came from; the embedded `recon/` sha is printed in cell 4 and pins
exactly which scripts ran.


In [ ]:
import shutil, zipfile, datetime

STAMP = datetime.date.today().isoformat()
bundle = pathlib.Path(f"/content/lingbotmap_colab_{STAMP}.zip")

KEEP = ["run.json", "scale.json", "clean_stats.json", "inspect_stats.json",
        "cloud_clean.ply", "trajectory.npz",
        "ate.json", "ate_plot.png"]          # Part 2: ground-truth scoring
if KEEP_RAW_PLY:
    KEEP.append("cloud.ply")

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for d in sorted((WORKP / "runs").iterdir()):
        for n in KEEP:
            if (d / n).exists():
                z.write(d / n, f"{d.name}/{n}")
        for q in d.glob("view_*.png"):
            z.write(q, f"{d.name}/{q.name}")
    for q in (WORKP / "logs").glob("*.log"):
        z.write(q, f"logs/{q.name}")
    # Part 2 provenance: which bytes the GrandTour runs actually used, and the preflight.
    for n in ("mission.json", "flow.json", "gt_tum.txt", "contact_sheet.jpg"):
        p_ = WORKP / "grandtour" / EIG_MISSION / n
        if p_.exists():
            z.write(p_, f"grandtour_{EIG_MISSION}/{n}")

print(f"{bundle}  {bundle.stat().st_size/1e6:.1f} MB")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = pathlib.Path("/content/drive/MyDrive/GeologicDome/colab_runs")
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(bundle, dst); print("copied to", dst)

gpu = P.name.replace("NVIDIA ", "")
print("\n" + "=" * 74 + "\npaste into notes/experiments.md:\n")
for tag in sorted(RUNS):
    r = RUNS[tag]
    scene = tag.split("_")[0]
    if r.get("flow_threshold"):
        how = (f"**VO/windowed ws={r['window_size']}**, flow {r['flow_threshold']:g} px / gap "
               f"{r['max_non_keyframe_gap']}, keyframes {r['n_keyframes']}/{r['n_frames']} "
               f"({r['keyframe_frac']:.0%})")
    else:
        how = f"Direct/streaming, kfi={r['keyframe_interval']}"
    cfg = (f"LingBot-Map base on **Colab {gpu} {VRAM_GB:.0f} GB**, upstream `example/{scene}` "
           f"({r['n_frames']} frames), {how}, kvsw={r['kv_cache_sliding_window']}, "
           f"nsf={r['num_scale_frames']}, 518 crop")
    met = (f"{r['inference_s']:.0f} s, {r['fps']:.2f} fps, peak VRAM {r['peak_vram_gb']:.2f} GB, "
           f"{r['n_points']:,} pts, **ratio {r['traj_length_over_extent']}**")
    print(f"| {STAMP}-colab-{tag.replace('_','-')} | <hash> | {cfg} | — | {met} | <takeaway> |")

from google.colab import files
files.download(str(bundle))